# Reinforcement Learning: Q-Learning

An agent learns to make decisions by interacting with an environment and receiving rewards.

1. **RL Framework** - Agent, environment, state, action, reward
2. **Q-Learning** - Learning action-value function via Bellman equation
3. **Exploration vs Exploitation** - Epsilon-greedy strategy
4. **Training an Agent** - Solve FrozenLake with Gymnasium

**Environment**: FrozenLake-v1 (Gymnasium)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
# pip install gymnasium
import gymnasium as gym

np.random.seed(42)

## 1. The RL Framework

At each timestep t:
1. Agent observes state $s_t$
2. Agent selects action $a_t$
3. Environment returns reward $r_t$ and next state $s_{t+1}$
4. Goal: maximize cumulative discounted reward $\sum_{t=0}^{\infty} \gamma^t r_t$

## Q-Learning Update Rule

$$Q(s,a) \leftarrow Q(s,a) + \alpha \left[ r + \gamma \max_{a'} Q(s', a') - Q(s,a) \right]$$

- $\alpha$ = learning rate
- $\gamma$ = discount factor (how much to value future rewards)
- $\max_{a'} Q(s', a')$ = best possible future value

In [ ]:
# Create FrozenLake environment
# The agent must navigate a frozen lake from Start (S) to Goal (G)
# without falling into Holes (H). Frozen (F) tiles are safe.
env = gym.make("FrozenLake-v1", map_name="4x4", is_slippery=False)

n_states = env.observation_space.n
n_actions = env.action_space.n

print(f"States: {n_states}, Actions: {n_actions}")
print("Actions: 0=Left, 1=Down, 2=Right, 3=Up")
print(f"\nMap:")
env.reset()
print(env.unwrapped.desc)

In [ ]:
# Q-Learning implementation
def q_learning(
    env, n_episodes=10000, alpha=0.1, gamma=0.99,
    epsilon_start=1.0, epsilon_end=0.01, epsilon_decay=0.9995
):
    """Train Q-learning agent."""
    q_table = np.zeros((n_states, n_actions))
    epsilon = epsilon_start
    rewards_per_episode = []
    
    for episode in range(n_episodes):
        state, _ = env.reset()
        total_reward = 0
        done = False
        
        while not done:
            # Epsilon-greedy action selection
            if np.random.random() < epsilon:
                action = env.action_space.sample()  # Explore
            else:
                action = np.argmax(q_table[state])  # Exploit
            
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            
            # Q-learning update
            best_next = np.max(q_table[next_state])
            q_table[state, action] += alpha * (
                reward + gamma * best_next * (1 - terminated) - q_table[state, action]
            )
            
            state = next_state
            total_reward += reward
        
        rewards_per_episode.append(total_reward)
        epsilon = max(epsilon_end, epsilon * epsilon_decay)
    
    return q_table, rewards_per_episode

q_table, rewards = q_learning(env)

# Plot learning curve
window = 100
rolling_avg = np.convolve(rewards, np.ones(window)/window, mode="valid")

plt.figure(figsize=(10, 4))
plt.plot(rolling_avg, color="teal")
plt.xlabel("Episode")
plt.ylabel(f"Average Reward ({window}-episode window)")
plt.title("Q-Learning Training Progress")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Final success rate (last 1000): {np.mean(rewards[-1000:])*100:.1f}%")

In [ ]:
# Visualize learned Q-table
action_symbols = ["<", "v", ">", "^"]

print("Learned Policy:")
for i in range(4):
    row = ""
    for j in range(4):
        state = i * 4 + j
        desc = env.unwrapped.desc[i][j].decode()
        if desc in ["H", "G"]:
            row += f"  {desc} "
        else:
            best_action = np.argmax(q_table[state])
            row += f"  {action_symbols[best_action]} "
    print(row)

print("\nQ-Table (rounded):")
print(np.round(q_table, 2))

In [ ]:
# Test the trained agent
def test_agent(env, q_table, n_episodes=100):
    successes = 0
    for _ in range(n_episodes):
        state, _ = env.reset()
        done = False
        while not done:
            action = np.argmax(q_table[state])
            state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
        successes += reward
    return successes / n_episodes

success_rate = test_agent(env, q_table)
print(f"Test success rate: {success_rate*100:.0f}%")

## Key Takeaways

1. **Q-learning is model-free** - the agent learns without knowing environment dynamics
2. **Epsilon-greedy balances explore/exploit** - start high (explore), decay over time (exploit)
3. **Discount factor gamma** controls short-term vs long-term thinking
4. **Q-tables only work for small state spaces** - for large/continuous spaces, use Deep Q-Networks (DQN)
5. **RL is sample-inefficient** - thousands of episodes needed even for simple environments
6. **State-of-the-art**: PPO, SAC, and TD3 are the modern policy gradient methods used in production